# 01：一步一步搭建你的第一个量子电路

从处于 `00` 状态的两个量子比特开始，每次添加一个量子门，观察变化，最后生成 Bell 态。
全部计算代码都在本 notebook 中，无需打开辅助脚本。
完成[环境准备](../SETUP.md)后，按 **Shift+Enter** 逐格运行。

[English edition](../notebooks/01_first_circuit.ipynb)

**学习目标：** 构造、画出并运行电路，然后读懂振幅与概率。

**先修要求：** 了解 Python 变量与函数调用即可，不需要 PyTorch 训练知识或云端账号。若量子态对你还很陌生，请先完成本课再进入 02。

学习路线见 [workshop 导航](../README.md)。


## 1. 导入框架

`fq` 是 FlagQuantum 的简称。如果不确定 notebook 使用的是哪一份安装，可以检查下方打印的模块路径。


In [ ]:
import os
import subprocess
from pathlib import Path

import flagquantum as fq

package_path = Path(fq.__file__).resolve()
source_commit = os.getenv("FLAGQUANTUM_SOURCE_COMMIT")
if not source_commit:
    try:
        source_commit = subprocess.run(
            ["git", "-C", str(package_path.parent), "rev-parse", "HEAD"],
            check=True,
            capture_output=True,
            text=True,
            timeout=5,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
        source_commit = "not recorded by this deployment"
print("FlagQuantum version:", fq.__version__)
print("FlagQuantum source:", package_path)
print("FlagQuantum commit:", source_commit)
print("QUAFU_API_KEY configured:", bool(os.getenv("QUAFU_API_KEY")))
print("QUAFU_TASK_SERVER_URL configured:", bool(os.getenv("QUAFU_TASK_SERVER_URL")))


## 2. 创建两个量子比特

新电路的每个量子比特都从 `0` 状态开始。创建电路是在描述程序，`fq.run` 才会执行它。

**运行前想一想：**`00`、`01`、`10` 和 `11` 中，哪一个结果的概率应为 1？

电路图中，每条横线对应一个量子比特，量子门从左到右执行。图中的上下顺序不等于云平台结果位串的排列顺序。


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

circuit = fq.Circuit(2)
fig, ax = circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
result = fq.run(circuit)
state = result.to_statevector().reshape(-1)
print("State amplitudes:", state)


态向量为每个可能的比特串记录一个复数振幅。振幅的绝对值平方就是对应结果的概率。
下方四个分量依次对应 `00`、`01`、`10` 和 `11`。

**如何读位串？** FlagQuantum 默认的全寄存器位串从左到右按 `q0、q1` 排列。因此 `10` 表示 q0=1、q1=0，`01` 表示 q0=0、q1=1。态向量索引中 q0 是最高有效位（大端约定）。三个量子比特时，顺序就是 `q0、q1、q2`。


In [ ]:
labels = ["00", "01", "10", "11"]
probabilities = state.abs().square()
print(dict(zip(labels, probabilities.tolist())))


**用非对称电路验证位序。** 只对 q0 加 X，结果应是 `10`，不是 `01`。这里使用独立电路，原电路仍保持 `00`，不影响下一步。


In [ ]:
order_check = fq.Circuit(2).x(0)
fig, ax = order_check.draw(format="mpl", show_all_qubits=True)
plt.show()
order_state = fq.run(order_check).to_statevector().reshape(-1)
order_probabilities = order_state.abs().square()
assert order_probabilities.tolist() == [0.0, 0.0, 1.0, 0.0]
print("Bit order: q0 q1 (left to right)")
print(dict(zip(labels, order_probabilities.tolist())))


## 3. 添加 Hadamard 门

`h(0)` 作用于量子比特 0，使它进入叠加态。量子比特 1 仍保持为零。
**先预测：**现在可能出现哪两个结果？


In [ ]:
circuit.h(0)
fig, ax = circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
result = fq.run(circuit)
state = result.to_statevector().reshape(-1)
print(dict(zip(labels, state.abs().square().tolist())))


## 4. 连接两个量子比特

当量子比特 0 为 `1` 时，`cx(0, 1)` 翻转量子比特 1。前面已经添加了 Hadamard 门，因此这一步会生成 Bell 态。
此时应只有 `00` 和 `11` 两种结果，各自的概率为 0.5。

添加量子门会修改电路对象。重复运行“添加门”的单元，会把同一个门再添加一次。
如果要重新开始，请从创建 `circuit` 的单元开始运行。

**概念：** 振幅可以是复数，概率是振幅绝对值的平方。概率表不保留振幅之间的相对相位。

**预期输出：** 这个理想电路给出 P(00)=P(11)=0.5；态向量还包含产生这些概率的振幅。

**常见误解：** 只看到 00 和 11 并不能单独证明纠缠。不相干的混合态也可以产生同样的分布；需要结合制备电路及额外测量来区分它与贝尔态。


In [ ]:
circuit.cx(0, 1)
fig, ax = circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
result = fq.run(circuit)
state = result.to_statevector().reshape(-1)
probabilities = state.abs().square()
print(dict(zip(labels, probabilities.tolist())))


## 5. 绘图并检查结果

将模拟得到的概率与前面推导出的答案比较，检查你的电路是否正确。

**读懂输出：** `{'00': 0.5, '01': 0.0, '10': 0.0, '11': 0.5}` 是理想概率表，微小数值差异是正常的。每根柱子代表概率，而不是实测计数；标签从左到右按 q0 q1 排列。

**安全重复运行：** 再次运行 `h(0)` 或 `cx(0, 1)` 会追加一个门。要重做相同实验，应从创建电路的单元开始；只重新运行 `fq.run` 会执行当前电路，不会追加门。


In [ ]:
import torch

expected = torch.tensor([0.5, 0.0, 0.0, 0.5])
torch.testing.assert_close(probabilities, expected, atol=1e-6, rtol=1e-6)
plt.bar(labels, probabilities.tolist())
plt.xlabel("Bit string (q0 q1)")
plt.ylabel("Probability")
plt.ylim(0, 1)
plt.show()


## 6. 尝试自己的电路

修改下方的 `use_entangling_gate`，比较概率的变化。这里每次都创建新电路，因此可以安全地重复运行。
模拟态给出的是精确概率；后面使用真实设备时，将通过多次执行并测量（称为 *shots*）估计这些概率。


In [ ]:
use_entangling_gate = False
my_circuit = fq.Circuit(2)
my_circuit.h(0)
if use_entangling_gate:
    my_circuit.cx(0, 1)
fig, ax = my_circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
my_state = fq.run(my_circuit).to_statevector().reshape(-1)
print(dict(zip(labels, my_state.abs().square().tolist())))


## 带回家使用

这是完整的本地 Bell 态示例。它只需要 FlagQuantum 和 PyTorch，
不依赖 workshop 目录、平台账号或特定镜像。

```python
import flagquantum as fq

bell = fq.Circuit(2).h(0).cx(0, 1)
probabilities = fq.run(bell).to_statevector().reshape(-1).abs().square()
print(probabilities.tolist())  # approximately [0.5, 0, 0, 0.5]
```
